In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil, os
shutil.copytree("/content/drive/MyDrive/IITD_Palmprint", "/content/IITD_Palmprint")

'/content/IITD_Palmprint'

In [ ]:
import numpy as np
import tensorflow as tf
import pickle
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
dataset_path = "/content/IITD_Palmprint"
img_size = 224
batch_size = 32
n_clusters = 16

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode=None,
    shuffle=False)

Found 5201 images belonging to 3 classes.


In [ ]:
full_model = load_model("/content/drive/MyDrive/palm_model.h5")

feature_model = Model(
    inputs=full_model.layers[0].input,
    outputs=full_model.get_layer('feature_layer').output)

In [ ]:
raw_features = feature_model.predict(generator)
raw_features = raw_features.reshape(raw_features.shape[0], -1)
print("Kích thước ma trận đặc trưng gốc:", raw_features.shape)

Đang trích xuất đặc trưng từ ảnh bàn tay...
163/163 ━━━━━━━━━━━━━━━━━━━━ 277s 2s/step
Kích thước ma trận đặc trưng gốc: (5201, 256)


In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(raw_features)

In [ ]:
print(f"Đang tiến hành gom cụm với K = {n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_features)
print("Chúc mừng! Đã gom cụm xong 16 nhóm vận mệnh.")

Đang tiến hành gom cụm với K = 16...
Chúc mừng! Đã gom cụm xong 16 nhóm vận mệnh.


In [ ]:
with open("/content/drive/MyDrive/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("/content/drive/MyDrive/kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans, f)